# MuJoCo leg kinematics in Colab (v2 — fixed camera framing)

This is the v2 revision of `leg_kinematics_colab.ipynb`. The only behavioral change is the camera definition in the model XML: it has been moved and widened so the leg stays inside the frame across the full knee sweep.

**What changed and why.** The original camera was `pos="1.0 -3.0 0.45" xyaxes="1 0 0 0 0 1" fovy="35"`. Because `xyaxes` makes the camera's local x-axis world `+x`, the image is centered horizontally on **world x = camera_x = 1.0**. The leg's pelvis sits at world x = 0 and the toe sweeps into negative x (toe_x goes from −0.005 at knee_deg=0° to −0.452 at knee_deg=90°). With `fovy=35°` and the camera 3 m away, the visible horizontal range at the leg's depth was roughly world x ∈ [−0.24, +2.24], so the foot drifted off the left edge once the knee bent past ~50°.

The v2 camera is `pos="-0.2 -2.5 0.5" xyaxes="1 0 0 0 0 1" fovy="45"`:
- `pos x = -0.2` centers the image on the **midpoint of the toe sweep** (between 0 and −0.45), so the leg stays centered the whole animation.
- `pos y = -2.5` brings the camera slightly closer and `fovy=45` widens the angle, giving a comfortable margin at extreme flexion.
- `pos z = 0.5` splits the vertical between the pelvis (z = 0.9) and the foot (z ≈ 0.06).

Everything else — the model geometry, the kinematics helper, the rendering loop — is identical to v1.

In [ ]:
# Colab setup. Run this first.
import os
import sys
import subprocess
from importlib.metadata import PackageNotFoundError, version

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mujoco",
        "mediapy",
        "matplotlib",
    ])

# MuJoCo needs a headless renderer in Colab.
os.environ.setdefault("MUJOCO_GL", "egl")

# If this notebook is run locally from D:/Robo/codex, the cloned
# source folder named "mujoco" can shadow the pip package.
cwd = os.getcwd()
sys.path = [path for path in sys.path if os.path.abspath(path or cwd) != cwd]

import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
import mujoco

try:
    mujoco_version = version("mujoco")
except PackageNotFoundError:
    mujoco_version = "unknown"

print("MuJoCo package version:", mujoco_version)

In [ ]:
LEG_XML = """
<mujoco model="planar_leg_kinematics">
  <compiler angle="degree" autolimits="true"/>
  <option gravity="0 0 -9.81" timestep="0.002"/>

  <default>
    <joint type="hinge" axis="0 1 0" damping="0.05"/>
    <geom type="capsule" density="500" contype="0" conaffinity="0"/>
    <site size="0.025"/>
  </default>

  <worldbody>
    <light pos="0 -3 3" dir="0 1 -1"/>
    <!-- v2: camera re-aimed so leg stays centered through knee sweep.
         Was: pos="1.0 -3.0 0.45" fovy="35" (centered image on world x=1.0,
         left edge clipped the foot once knee_deg > ~50). Now: pos x=-0.2
         centers on the toe-sweep midpoint, fovy=45 widens the frame. -->
    <camera name="side" pos="-0.2 -2.5 0.5" xyaxes="1 0 0 0 0 1" fovy="45"/>

    <body name="pelvis" pos="0 0 0.90">
      <geom name="pelvis_geom" type="sphere" size="0.055" rgba="0.20 0.20 0.22 1"/>
      <site name="hip_site" pos="0 0 0" rgba="0.15 0.25 1.0 1"/>

      <body name="thigh" pos="0 0 0">
        <joint name="hip" range="-90 90"/>
        <geom name="thigh_geom" fromto="0 0 0 0 0 -0.42" size="0.035" rgba="0.20 0.45 0.90 1"/>

        <body name="shank" pos="0 0 -0.42">
          <joint name="knee" range="0 135"/>
          <site name="knee_site" pos="0 0 0" rgba="1.0 0.55 0.0 1"/>
          <geom name="shank_geom" fromto="0 0 0 0 0 -0.42" size="0.030" rgba="0.10 0.75 0.45 1"/>

          <body name="foot" pos="0 0 -0.42">
            <joint name="ankle" range="-45 45"/>
            <site name="ankle_site" pos="0 0 0" rgba="1.0 0.0 0.0 1"/>
            <geom name="foot_geom" fromto="-0.06 0 0 0.22 0 0" size="0.025" rgba="0.95 0.25 0.25 1"/>
            <site name="toe_site" pos="0.22 0 0" rgba="0.80 0.0 1.0 1"/>
          </body>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(LEG_XML)
data = mujoco.MjData(model)

JOINT_NAMES = ["hip", "knee", "ankle"]
SITE_NAMES = ["hip_site", "knee_site", "ankle_site", "toe_site"]

joint_qpos_addr = {}
for joint_name in JOINT_NAMES:
    joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
    joint_qpos_addr[joint_name] = model.jnt_qposadr[joint_id]

site_id = {
    site_name: mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name)
    for site_name in SITE_NAMES
}

print("qpos size:", model.nq)
print("joints:", joint_qpos_addr)

In [ ]:
def set_leg_pose(hip_deg=0.0, knee_deg=0.0, ankle_deg=0.0):
    """Set joint angles in degrees and return site positions in world coordinates."""
    angles_deg = {
        "hip": hip_deg,
        "knee": knee_deg,
        "ankle": ankle_deg,
    }

    data.qpos[:] = 0.0
    data.qvel[:] = 0.0

    for joint_name, angle_deg in angles_deg.items():
        data.qpos[joint_qpos_addr[joint_name]] = np.deg2rad(angle_deg)

    mujoco.mj_forward(model, data)
    return {
        site_name: data.site_xpos[site_id[site_name]].copy()
        for site_name in SITE_NAMES
    }


def print_site_positions(points):
    for site_name in SITE_NAMES:
        x, y, z = points[site_name]
        print(f"{site_name:10s}  x={x: .3f} m  y={y: .3f} m  z={z: .3f} m")


def plot_leg(points, title=None):
    x = [points[name][0] for name in SITE_NAMES]
    z = [points[name][2] for name in SITE_NAMES]

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(x, z, marker="o", linewidth=4)
    for name, px, pz in zip(SITE_NAMES, x, z):
        ax.text(px + 0.015, pz + 0.015, name.replace("_site", ""))

    ax.set_xlabel("x position (m)")
    ax.set_ylabel("z position (m)")
    ax.set_xlim(-0.60, 0.60)
    ax.set_ylim(0.00, 1.05)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    if title:
        ax.set_title(title)
    plt.show()

In [ ]:
pose = set_leg_pose(hip_deg=20, knee_deg=65, ankle_deg=-20)
print_site_positions(pose)
plot_leg(pose, "Forward kinematics: hip=20 deg, knee=65 deg, ankle=-20 deg")

In [ ]:
def render_current_pose(width=640, height=480):
    with mujoco.Renderer(model, height=height, width=width) as renderer:
        renderer.update_scene(data, "side")
        return renderer.render()


set_leg_pose(hip_deg=20, knee_deg=65, ankle_deg=-20)
image = render_current_pose()
media.show_image(image)

In [ ]:
# Sweep the knee angle and track the toe position.
rows = []
frames = []

with mujoco.Renderer(model, height=480, width=640) as renderer:
    for knee_deg in np.linspace(0, 110, 40):
        ankle_deg = -0.35 * knee_deg
        points = set_leg_pose(hip_deg=15, knee_deg=knee_deg, ankle_deg=ankle_deg)
        toe = points["toe_site"]
        rows.append((knee_deg, ankle_deg, toe[0], toe[1], toe[2]))

        renderer.update_scene(data, "side")
        frames.append(renderer.render().copy())

print("knee_deg  ankle_deg  toe_x  toe_y  toe_z")
for row in rows[::8]:
    print(f"{row[0]:8.1f}  {row[1]:9.1f}  {row[2]: .3f}  {row[3]: .3f}  {row[4]: .3f}")

media.show_video(frames, fps=12)